# Install packages

In [1]:
!pip install ase scikit-learn scipy numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.3 MB/s eta 0:00:00


In [3]:
!pip install pymatgen -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.

# Checking clean slab's geometry

In [10]:
# --- Symmetry check on the clean, relaxed Co slab -----------------------
# Purpose: confirm the slab is symmetric enough that all candidate sites of
# a given type (ontop, bridge, hollow_fcc, hollow_hcp) are physically
# equivalent, so it's valid to run VASP on just one representative site per
# type instead of every candidate found by the Delaunay site-finder.

slab = read("/content/CONTCAR_Co")  # relaxed, bare Co surface (no adsorbate)

from pymatgen.io.ase import AseAtomsAdaptor        # ASE <-> pymatgen structure converter
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer  # pymatgen's symmetry detector

# SpacegroupAnalyzer only accepts pymatgen Structure objects, so convert first
pmg_slab = AseAtomsAdaptor.get_structure(slab)

# symprec = tolerance (Angstrom) for how far an atom can sit from its "ideal"
# symmetric position and still be treated as symmetric. Set loosely (0.1 A)
# because this is a DFT-relaxed slab, not a perfect bulk-truncated structure.
sga = SpacegroupAnalyzer(pmg_slab, symprec=0.1)

# Space group symbol: describes the slab's overall symmetry (rotations,
# mirrors, etc). A high-symmetry result (e.g. P3m1) supports reducing
# candidate adsorption sites down to one per type. A low-symmetry result
# (e.g. P1) means the surface isn't uniform and more sites should be kept.
print("Space group:", sga.get_space_group_symbol())

# Number of symmetry operations found (rotations/reflections/translations
# that map the slab onto itself). More operations = more redundancy among
# the 96 candidate sites found earlier.
print("Point group ops:", len(sga.get_symmetry_operations()))

Space group: P3m1
Point group ops: 32


# Cleaning directory from previous run

In [39]:
import shutil
shutil.rmtree("/content/adsorption_sites", ignore_errors=True)
# Run this cell before the main generation cell each time you
# rerun it — it deletes /content/adsorption_sites
#(and everything in it) so you start clean instead of accumulating old
#folders from previous adsorbates

# Placing adsorbate species on surafce

In [37]:
import os
import shutil
import numpy as np
from ase.io import read, write
from ase.constraints import FixAtoms
from sklearn.cluster import KMeans
from scipy.spatial import Delaunay

SLAB_FILE = "/content/CONTCAR_Co"
ADS_FILE = "/content/CONTCAR_SH2"
N_LAYERS = 5
N_FIXED_LAYERS = 2
HEIGHT = 1.8
NN_CUTOFF = 3.0
HCP_THRESH = 0.75
SEED = 42
OUT_ROOT = "/content/adsorption_sites"

# Derive adsorbate label from the CONTCAR filename, e.g. CONTCAR_SH2 -> "SH2"
ADS_LABEL = os.path.basename(ADS_FILE).replace("CONTCAR_", "")

os.makedirs(OUT_ROOT, exist_ok=True)

def get_layer_indices(atoms, n_layers, seed):
    z = atoms.positions[:, 2].reshape(-1, 1)
    km = KMeans(n_clusters=n_layers, random_state=seed, n_init=10).fit(z)
    means = [atoms.positions[km.labels_ == i, 2].mean() for i in range(n_layers)]
    order = np.argsort(means)[::-1]
    return [np.where(km.labels_ == order[k])[0] for k in range(n_layers)]

def periodic_xy(xy, cell, reps=(-1, 0, 1)):
    a, b = cell[0][:2], cell[1][:2]
    return np.vstack([xy + i*np.array(a) + j*np.array(b) for i in reps for j in reps])

def in_central_cell(xy, cell, tol=1e-3):
    A = np.array([cell[0][:2], cell[1][:2]]).T
    frac = np.linalg.solve(A, xy)
    return np.all(frac >= -tol) and np.all(frac < 1 - tol)

def find_sites(atoms, layers):
    top_idx, second_idx = layers[0], layers[1]
    z_surf = atoms.positions[top_idx, 2].mean()
    top_xy = atoms.positions[top_idx, :2]
    second_xy = periodic_xy(atoms.positions[second_idx, :2], atoms.get_cell())

    sites = [dict(type="ontop", xy=xy) for xy in top_xy]

    tiled = periodic_xy(top_xy, atoms.get_cell())
    tri = Delaunay(tiled)

    seen = set()
    for simplex in tri.simplices:
        for i in range(3):
            a, b = simplex[i], simplex[(i+1) % 3]
            key = tuple(sorted((a, b)))
            if key in seen:
                continue
            seen.add(key)
            p1, p2 = tiled[a], tiled[b]
            if np.linalg.norm(p1 - p2) > NN_CUTOFF:
                continue
            mid = (p1 + p2) / 2
            if in_central_cell(mid, atoms.get_cell()):
                sites.append(dict(type="bridge", xy=mid))

    for simplex in tri.simplices:
        pts = tiled[simplex]
        edges = [np.linalg.norm(pts[i]-pts[(i+1) % 3]) for i in range(3)]
        if max(edges) > NN_CUTOFF:
            continue
        c = pts.mean(axis=0)
        if not in_central_cell(c, atoms.get_cell()):
            continue
        d = np.linalg.norm(second_xy - c, axis=1)
        t = "hollow_hcp" if d.min() < HCP_THRESH else "hollow_fcc"
        sites.append(dict(type=t, xy=c))

    for s in sites:
        s["z"] = z_surf
    return sites

def dedupe(sites, tol=0.3):
    kept = []
    for s in sites:
        if not any(k["type"] == s["type"] and np.linalg.norm(np.array(k["xy"])-np.array(s["xy"])) < tol for k in kept):
            kept.append(s)
    return kept

def find_anchor_index(atoms, symbol):
    idx = [i for i, a in enumerate(atoms) if a.symbol == symbol]
    if len(idx) != 1:
        raise ValueError(f"Expected exactly 1 {symbol} atom, found {len(idx)}")
    return idx[0]

ANCHOR_SYMBOL = "S"  # change if your adsorbate binds through a different atom

def place_adsorbate(slab, ads, anchor_idx, xy, z, height):
    mol = ads.copy()
    n = len(mol)
    if n > 1:
        anchor_pos = mol.positions[anchor_idx].copy()
        others = [i for i in range(n) if i != anchor_idx]
        tail = mol.positions[others].mean(axis=0) - anchor_pos
        tn = np.linalg.norm(tail)
        if tn > 1e-6:
            tail /= tn
            target = np.array([0.0, 0.0, 1.0])
            axis = np.cross(tail, target)
            an = np.linalg.norm(axis)
            if an > 1e-6:
                axis /= an
                angle = np.degrees(np.arccos(np.clip(np.dot(tail, target), -1, 1)))
                mol.translate(-anchor_pos)
                mol.rotate(angle, axis)
                mol.translate(anchor_pos)

    mol.translate(np.array([xy[0], xy[1], z + height]) - mol.positions[anchor_idx])
    combined = slab.copy() + mol
    n_slab = len(slab)

    guard = 0
    while guard < 50:
        d = combined.get_all_distances()
        if np.min(d[n_slab:, :n_slab]) >= 1.0:
            break
        combined.positions[n_slab:, 2] += 0.1
        guard += 1
    return combined

slab = read(SLAB_FILE)
adsorbate = read(ADS_FILE)
anchor_idx = find_anchor_index(adsorbate, ANCHOR_SYMBOL)

layers = get_layer_indices(slab, N_LAYERS, SEED)
fixed = np.concatenate(layers[-N_FIXED_LAYERS:]).tolist()

sites = dedupe(find_sites(slab, layers))
by_type = {}
for s in sites:
    by_type.setdefault(s["type"], []).append(s)

for site_type, site_list in by_type.items():
    print(f"  {site_type}: {len(site_list)} symmetry-equivalent candidate(s) found")

# Slab confirmed as high-symmetry (P3m1, 32 ops) via SpacegroupAnalyzer check,
# so all candidates of a given site type are physically equivalent.
# Keep just one representative per type instead of writing every candidate.
chosen_sites = {t: s[0] for t, s in by_type.items()}

for site_type, site in chosen_sites.items():
    system = place_adsorbate(slab, adsorbate, anchor_idx, site["xy"], site["z"], HEIGHT)
    system.set_constraint(FixAtoms(indices=fixed))
    folder = os.path.join(OUT_ROOT, f"{ADS_LABEL}_{site_type}")
    os.makedirs(folder, exist_ok=True)
    write(os.path.join(folder, "POSCAR"), system, format="vasp", vasp5=True, direct=True)
    print(f"wrote {folder}/POSCAR")

# --- Zip everything and download to your local machine ---
zip_path = f"/content/{ADS_LABEL}_adsorption_sites"
shutil.make_archive(zip_path, "zip", OUT_ROOT)

from google.colab import files
files.download(f"{zip_path}.zip")

  ontop: 16 symmetry-equivalent candidate(s) found
  bridge: 48 symmetry-equivalent candidate(s) found
  hollow_fcc: 16 symmetry-equivalent candidate(s) found
  hollow_hcp: 16 symmetry-equivalent candidate(s) found
wrote /content/adsorption_sites/SH2_ontop/POSCAR
wrote /content/adsorption_sites/SH2_bridge/POSCAR
wrote /content/adsorption_sites/SH2_hollow_fcc/POSCAR
wrote /content/adsorption_sites/SH2_hollow_hcp/POSCAR


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Checking final adsorption site after DFT

In [ ]:
import glob
import re
import numpy as np
from ase.io import read
from ase.neighborlist import NeighborList, natural_cutoffs

METAL_SYMBOL = "Co"
ANCHOR_SYMBOL = "S"          # same as your placement anchor
CONTCAR_PATTERN = "/content/CONTCAR_Co_*"   # matches CONTCAR_Co_H2S_ontop, CONTCAR_Co_H2S_bridge, ...
POSCAR_ROOT = "/content/adsorption_sites"   # your pre-VASP POSCARs, for dissociation comparison
COORD_CUTOFF = 3.2           # distance (A) under which a metal atom counts as "bonded" to the anchor

def check_dissociation(initial_ads, final_ads):
    try:
        inl = NeighborList(natural_cutoffs(initial_ads), self_interaction=False, bothways=True)
        inl.update(initial_ads)
        fnl = NeighborList(natural_cutoffs(final_ads), self_interaction=False, bothways=True)
        fnl.update(final_ads)
        for i in range(len(initial_ads)):
            if set(inl.get_neighbors(i)[0]) != set(fnl.get_neighbors(i)[0]):
                return True
        return False
    except Exception:
        return True

def check_desorption(system, n_slab_atoms, cushion=1.5):
    try:
        cutoffs = [c * cushion for c in natural_cutoffs(system)]
        nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
        nl.update(system)
        for ads_idx in range(n_slab_atoms, len(system)):
            if any(n < n_slab_atoms for n in nl.get_neighbors(ads_idx)[0]):
                return False
        return True
    except Exception:
        return True

def parse_label_site(path):
    # e.g. CONTCAR_Co_H2S_ontop -> label="H2S", site="ontop"
    fname = path.split("/")[-1]
    m = re.match(r"CONTCAR_Co_(.+)_(ontop|bridge|hollow_fcc|hollow_hcp)$", fname)
    if not m:
        raise ValueError(f"Filename doesn't match expected pattern: {fname}")
    return m.group(1), m.group(2)

def analyze_final_structure(contcar_path):
    ads_label, site_label = parse_label_site(contcar_path)
    final_system = read(contcar_path)

    # matching pre-VASP POSCAR, for the dissociation check and slab atom count
    poscar_path = f"{POSCAR_ROOT}/{ads_label}_{site_label}/POSCAR"
    initial_system = read(poscar_path)
    n_slab = sum(1 for a in initial_system if a.symbol == METAL_SYMBOL)

    initial_ads = initial_system[n_slab:].copy()
    final_ads = final_system[n_slab:].copy()

    dissociated = check_dissociation(initial_ads, final_ads) if len(initial_ads) > 1 else False
    desorbed = check_desorption(final_system, n_slab)

    anchor_idx = [i for i, a in enumerate(final_system) if a.symbol == ANCHOR_SYMBOL][0]
    metal_idx = [i for i, a in enumerate(final_system) if a.symbol == METAL_SYMBOL]

    top_metal_z = final_system.positions[metal_idx, 2].max()
    height = final_system.positions[anchor_idx, 2] - top_metal_z
    dists = sorted(final_system.get_distances(anchor_idx, metal_idx, mic=True))
    n_coord = sum(d < COORD_CUTOFF for d in dists)

    if n_coord == 1:
        final_site_type = "ontop"
    elif n_coord == 2:
        final_site_type = "bridge"
    elif n_coord >= 3:
        final_site_type = "hollow (fcc/hcp not distinguished here)"
    else:
        final_site_type = "desorbed / undercoordinated"

    return {
        "file": contcar_path.split("/")[-1],
        "intended_site": site_label,
        "final_site_by_coordination": final_site_type,
        "height_above_surface": round(height, 3),
        "anchor_metal_dist": round(dists[0], 3),
        "coordination_number": n_coord,
        "dissociated": dissociated,
        "desorbed": desorbed,
        "site_moved": site_label not in final_site_type,
    }

results = []
for path in sorted(glob.glob(CONTCAR_PATTERN)):
    try:
        results.append(analyze_final_structure(path))
    except Exception as e:
        print(f"Skipping {path}: {e}")

for r in results:
    print(r)